# 12F-01 — EPA source ingest and grain

**Question:** What exactly is in the EPA source, and what does one row represent?

This notebook is exploratory. Production ETL remains authoritative.

## Imports and portable paths

In [7]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
EPA_FILE = REPO_ROOT / "etl/data/raw/epa_testcar/epa_testcar_2026_raw.xlsx"
assert EPA_FILE.exists(), EPA_FILE


## Load the EPA source

No cleaning yet. First I want to see the workbook as delivered.

In [8]:
epa = pd.read_excel(EPA_FILE, sheet_name="Sheet1", engine="openpyxl")
print(f"{epa.shape[0]:,} rows × {epa.shape[1]} columns")


ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

## Shape, model years, and relevant columns

In [ ]:
year_counts = epa["Model Year"].value_counts(dropna=False).sort_index()
year_counts


In [ ]:
identity_columns = [
    "Model Year", "Represented Test Veh Make", "Represented Test Veh Model",
    "Actual Tested Testgroup", "Test Number", "Test Vehicle ID",
    "Test Veh Configuration #", "Test Procedure Description",
]
assert not [column for column in identity_columns if column not in epa.columns]
epa[identity_columns].head(8)


## First grain counts

Make+Model+Year is a commercial identity, not necessarily one test or technical state.

In [ ]:
mmy = ["Model Year", "Represented Test Veh Make", "Represented Test Veh Model"]
test_key = mmy + ["Actual Tested Testgroup", "Test Number"]
config_key = test_key + ["Test Vehicle ID", "Test Veh Configuration #"]

grain_counts = pd.Series({
    "source_rows": len(epa),
    "make_model_year": len(epa[mmy].drop_duplicates()),
    "test_key": len(epa[test_key].drop_duplicates()),
    "test_and_configuration_key": len(epa[config_key].drop_duplicates()),
})
grain_counts


## Real multi-state examples

In [ ]:
mmy_states = (
    epa.groupby(mmy, dropna=False)
       .agg(source_rows=("Model Year", "size"),
            test_groups=("Actual Tested Testgroup", "nunique"),
            configurations=("Test Veh Configuration #", "nunique"))
       .reset_index()
)
examples = mmy_states.query("test_groups > 1 or configurations > 1").sort_values("source_rows", ascending=False)
examples.head(10)


In [ ]:
example = examples.iloc[0]
same_vehicle = (
    epa["Model Year"].eq(example["Model Year"])
    & epa["Represented Test Veh Make"].eq(example["Represented Test Veh Make"])
    & epa["Represented Test Veh Model"].eq(example["Represented Test Veh Model"])
)
epa.loc[same_vehicle, identity_columns].head(20)


## What did we learn?

- One source row is a reported test/result row, not automatically one vehicle or one RUN.
- Make+Model+Year collapses distinct test groups and configurations.
- Next: keep this grain distinction visible while exploring non-BEV fuel and CO2 results.